In [ ]:
import pandas as pd

encoding = 'utf-8'

pd.set_option('max_colwidth',100)
pd.set_option('float_format', lambda x: '%.4f' % x)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 15)

In [ ]:
### Function:
### Judge the tweet time (in 3 years of pubtime)
###

def judge_year(df,year):
    for  index, row in df.iterrows():
        list_year =[i[:4] for i in row["twtime_list"]]
        cohort_year=[str(int(row["cohort"])-1),str(int(row["cohort"]))]
        if year==3:
            cohort_year=[str(int(row["cohort"])-1),str(int(row["cohort"])),str(int(row["cohort"])+1),str(int(row["cohort"])+2)]
            

        if bool(set(list_year) & set(cohort_year)): 
            df.at[index, 'with_tw_'+str(year)+'y'] = 1
            
        else: 
            df.at[index, 'with_tw_'+str(year)+'y'] = 0
            
            
    return df


### Function:
### revise the first name by removing "-" and "."
###
def name_revision(name_set):
    name_set[["last_name",'first_name']]=name_set[["last_name",'first_name']].fillna("NULL")
    name_set["first_name_revised"]=name_set["first_name"].apply(lambda x: x.replace("-"," ").replace("."," ")).apply(lambda x: re.split(" ",x)[0] if ((" " in x)& (len(re.split(" ",x)[0])>1)) else x)
    
    return name_set

In [ ]:
### read datasets : 

### 1. slelected early-career publications (first 3 years)
### 2. tweets info for all early-career publications
### 3. 

author_12_16_early_paper=pd.read_csv("...\\result\\0_author_doi_3y_2012_2016.csv",encoding=encoding)
author_12_16_early_paper["doi_lower"]=author_12_16_early_paper["doi"].apply(lambda x:str(x).lower())

author_12_16_early_paper_tw=pd.read__pickle("...\\result\\1_doi_tweet_info_all.csv")
author_12_16_early_paper_tw["len_tweet"]=author_12_16_early_paper_tw["Original_Tweet_list"].apply(lambda x:len(x) if type(x)==list else 0)
author_12_16_early_paper_tw["len_tweeter"]=author_12_16_early_paper_tw["Original_Tweeter_list"].apply(lambda x:len(x) if type(x)==list else 0)



### processing the data

#### Step 1: processing the authors names of publications with tweets
#### Step 2: maping the authors and tweeters

In [ ]:
### author name form publications


author_12_16_name=author_12_16_early_paper[['author_id',"first_name",'last_name']].groupby('author_id').\
                    agg({'last_name':lambda x: list(x),'first_name':lambda x: list(x),}.reset_index()     

author_12_16_name["last_name"]=author_12_16_name["last_name"].apply(lambda x:list(dict.fromkeys(x)))
author_12_16_name["first_name"]=author_12_16_name["first_name"].apply(lambda x:list(dict.fromkeys(x)))

author_12_16_name["last_name"]=author_12_16_name["last_name"].apply(lambda x:[y for y in x if pd.isna(y)==False])
author_12_16_name["first_name"]=author_12_16_name["first_name"].apply(lambda x:[y for y in x if pd.isna(y)==False])

author_12_16_name["last_name"]=author_12_16_name["last_name"].apply(lambda x:x[0] if len(x) else "@")
author_12_16_name["first_name"]=author_12_16_name["first_name"].apply(lambda x:x[0] if len(x) else "@")

In [ ]:
## the pub with tweets
author_12_16_tw_noNan=author_12_16_early_paper_tw[author_12_16_early_paper_tw["len_tweet"]>0]

author_12_16_tw_noNan=author_12_16_tw_noNan.merge(author_12_16_name,on="author_id",how="left")

author_12_16_tw_noNan=author_12_16_tw_noNan.drop_duplicates(['author_id',"doi"])

author_12_16_tw_noNan[["last_name",'first_name']]=author_12_16_tw_noNan[["last_name",'first_name']].fillna("NULL")
author_12_16_tw_noNan["first_name_revised"]=author_12_16_tw_noNan["first_name"].apply(lambda x: x.replace("-"," ").replace("."," ")).apply(lambda x: re.split(" ",x)[0] if ((" " in x)& (len(re.split(" ",x)[0])>1)) else x)


In [ ]:
## mapping the author and tweeter


author_tweeter=author_12_16_early_paper_tw[(author_12_16_early_paper_tw["Original_Tweeter_list"].notnull()) & (author_12_16_early_paper_tw["len_tweeter"]>0)]
author_tweeter["len_tweeterid_nonan"]=author_12_16_tw["Original_Tweeter_list"].apply(lambda x: len([y for y in x if pd.notnull(y)]))

author_tweeter=author_tweeter[author_tweeter["len_tweeterid_nonan"]>0][["author_id","doi","Original_Tweeter_list"]].explode('Original_Tweeter_list')


### self-promotion detection

#### compare author names and tweet names
#### ****more details can be found in the paper

In [ ]:
### Function:
### compare two names and return the similarity scores
###

from difflib import SequenceMatcher

titles = ['mr ', 'mr. ', 'dr. ', 'dr ', 'mrs ', 'mrs. ', 'prof. ', 'prof ', 'ms ', 'ms. ',"phd. ","phd " ]


def process_str(x):
    if "-" in x:
        x=x.replace("-"," ")
    x=x.lower()
    for item in titles:
        x= x.replace(item, '')
    
    return x

def similar(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def similar_x(a, b):
    try:
        a=re.sub('[^A-Za-z0-9]+', '', a)
        b=re.sub('[^A-Za-z0-9]+', '', b)
        score=SequenceMatcher(None, a.lower(), b.lower()).ratio()
    except:
        score=0
    return score

def compare_multi(f,l,fl,lf,f_l,l_f,l1):
    l1=process_str(l1)
    if " " in l1:
        simi_1_f_l=similar(f_l,l1)
        simi_1_l_f=similar(l_f,l1)
        simi_1_max=max(simi_1_f_l,simi_1_l_f)
        
    elif l1!="Null": 
        simi_1_f=similar(f,l1)
        simi_1_l=similar(l,l1)        
        simi_1_fl=similar(fl,l1)
        simi_1_lf=similar(lf,l1)
        simi_1_max=max(simi_1_f,simi_1_l,simi_1_fl,simi_1_lf)
        
    else:
        simi_1_max=0
    
    return simi_1_max


def self_verify(first,last,list1,list2,list3,listid,listT,key): 
    
    score_1=0
    score_2=0
    score_3=0
    tweeter_id_1=1
    tweeter_id_2=2
    tweeter_id_3=3
    author_name="NULL"
    user_name="NULL"
    handle="NULL"
    first=first.replace("@","")
    last=last.replace("@","")
    first_last=first+" "+last
    firstlast=first+last
    last_first=last+" "+first
    lastfirst=last+first
    
    
    for i in list1:
        if type(i)==list:
            for j in i:
                if type(j)==str:
                    simi_1_max=compare_multi(first,last,firstlast,lastfirst,first_last,last_first,j)

                    if (simi_1_max >score_1):
                        score_1=simi_1_max
                        author_name=j
                        tweeter_id_1=listT[list1.index(i)]
                        tweet_id_1=listid[list1.index(i)]

    for i in range(len(list2)):
         if type(list2[i])==str:
       
            simi_2_max=compare_multi(first,last,firstlast,lastfirst,first_last,last_first,list2[i])  
            if (simi_2_max >score_2):
                score_2=simi_2_max    
                user_name=list2[i]
                if key==0:
                    tweeter_id_2=listT[i]
                    tweet_id_2=listid[i]
                else:
                    tweeter_id_2=2 

                
    for i in range(len(list3)):
         if type(list3[i])==str:
            simi_3_max=compare_multi(first,last,firstlast,lastfirst,first_last,last_first,list3[i]) 
            if (simi_3_max >score_3):
                score_3=simi_3_max
                handle=list3[i]
                if key==0:
                    tweeter_id_3=listT[i]
                else:
                    tweeter_id_3=3 
        
    max_socre=max([score_1,score_2,score_3])
    if max_socre==score_1:
        attr="author_name_self_score"
        likely_name=author_name
        tweeter_id=tweeter_id_1
    elif max_socre==score_2:
        attr="tw_name_self_score"
        likely_name=user_name
        tweeter_id=tweeter_id_2
    else:
        attr="tw_handle_self_score"
        likely_name=handle
        tweeter_id=tweeter_id_3
                
    return score_1, score_2,score_3, author_name,user_name,handle,attr,max_socre,likely_name,tweeter_id


In [ ]:
def name_check(first, last, most_likely): 
    f_L=first+" "+ last
    L_f=last+" "+first
    
    if (match_name(f_L, most_likely)|match_name(L_f, most_likely)):
        return True
    
    else:
        return False
    
    
def intersection(lst1, lst2):
    lst3 = [value for value in lst1 if value in lst2]
    return lst3

In [ ]:
### data for comparision 1: 

author_name_openalex=pd.read_pickle("...\\3_1_openalex_tweeterid_name_processed_March.csv")

Tweeter_list=author_tweeter.merge(author_name_openalex,left_on="Original_Tweeter_int",right_on="tweeter_id",how="left")

Tweeter_list['author_name']=Tweeter_list['author_name'].fillna("0")
Tweeter_list=Tweeter_list.groupby(['author_id',"doi"])['author_name'].apply(list).reset_index()


author_12_16_tw_noNan=author_12_16_tw_noNan.merge(Tweeter_list,on=["author_id","doi"],how="left")

In [ ]:
author_12_16_tw_noNan["author_name_self_score"],author_12_16_tw_noNan["tw_name_self_score"],author_12_16_tw_noNan["tw_handle_self_score"],author_12_16_tw_noNan["likely_author"],author_12_16_tw_noNan["likely_name"],author_12_16_tw_noNan["likely_handle"],author_12_16_tw_noNan["max_attribute"],author_12_16_tw_noNan["max_score"],author_12_16_tw_noNan["most_likely_name"],author_12_16_tw_noNan["likely_Tweeter_ID"]= zip(*author_12_16_tw_noNan.apply(lambda row: self_verify(row["first_name_revised"],row["last_name"],row["author_name"],row["tw_name_list"],row["tw_handle_list"],row["Original_Tweet_list"],row["Original_Tweeter_list"],row["searched"]),axis=1))  


In [ ]:
### return the pairs with score over 0.6 

author_12_16_tw_noNan[(author_12_16_tw_noNan["max_score"]>0.6)].author_id.nunique()/author_12_16_tw_noNan.author_id.nunique()